In [1]:
# 访问参数/参数初始化/共享参数

In [4]:
# 单隐藏层
import torch
from torch import nn
net = nn.Sequential(nn.Linear(4,8),nn.ReLU(),nn.Linear(8,1))
X=torch.rand(size=(2,4)) #0-1均匀分布
print(X)
net(X)

tensor([[0.7579, 0.8626, 0.9123, 0.7083],
        [0.0582, 0.8717, 0.1418, 0.4086]])


tensor([[0.6760],
        [0.5437]], grad_fn=<AddmmBackward0>)

In [5]:
# 参数访问
# 通过Sequential类定义模型时，我们可以通过索引来访问模型的任意层
#  检查第二个全连接层的参数
print(net[2].state_dict())
#这个全连接层包含两个参数，分别是该层的权重和偏置。两者都存储为单精度浮点数（ffoat32）。注意，参数名称允许唯一标识每个参数

OrderedDict({'weight': tensor([[ 0.0431,  0.3006, -0.2469, -0.0185,  0.0497,  0.1893,  0.0094, -0.0384]]), 'bias': tensor([0.3495])})


In [8]:
# 要对参数执行任何操作，首先我们需要访问底层的数值
print(type(net[2].bias))
print(net[2].bias)
print(net[2].bias.data)

<class 'torch.nn.parameter.Parameter'>
Parameter containing:
tensor([0.3495], requires_grad=True)
tensor([0.3495])


In [9]:
net[2].weight.grad == None
# 还可以访问每个参数的梯度。在上面这个网络中，由于我们还没有调用反向传播，所以参数的梯度处于初始状态

True

In [10]:
# 访问第一个全连接层的参数和访问所有层
print(*[(name, param.shape) for name, param in net[0].named_parameters()])
print(*[(name, param.shape) for name, param in net.named_parameters()])

('weight', torch.Size([8, 4])) ('bias', torch.Size([8]))
('0.weight', torch.Size([8, 4])) ('0.bias', torch.Size([8])) ('2.weight', torch.Size([1, 8])) ('2.bias', torch.Size([1]))


In [11]:
#另一种访问网络参数的方式
net.state_dict()['2.bias'].data


tensor([0.3495])

In [12]:
# 从嵌套块收集参数
def block1():
    return nn.Sequential(nn.Linear(4,8),nn.ReLU(),
                        nn.Linear(8,4),nn.ReLU())

def block2():
    net = nn.Sequential() #顺序容器
    for i in range(4):
        net.add_module(f'block{i}',block1())
    return net
rgnet = nn.Sequential(block2(),nn.Linear(4,1))
rgnet(X)


tensor([[0.4164],
        [0.4163]], grad_fn=<AddmmBackward0>)

In [13]:
print(rgnet)


Sequential(
  (0): Sequential(
    (block0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [14]:
rgnet[0][1][0].bias.data
 # 访问第一个主要的块中、第二个子块的第一层的偏置项。


tensor([-0.2396,  0.3946, -0.2737,  0.0147, -0.1922,  0.4224,  0.1458,  0.0731])

In [16]:
## 参数初始化
# 内置的，将所有权重参数初始化为标准差为0.01的高斯随机变量，且将偏置参数设置为0
def init_normal(m):
    if type(m)==nn.Linear:
        nn.init.normal_(m.weight,mean=0,std=0.01)
        nn.init.zeros_(m.bias)
net.apply(init_normal)
net[0].weight.data[0],net[0].bias.data[0]
print(net.type)

<bound method Module.type of Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)>


In [17]:
#参数初始化为给定的常数
def init_constant(m): #m是将要被apply调用时传入的对象，net会循环调用每一层'lx'到init_c()中，m即lx
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 1)
        nn.init.zeros_(m.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [18]:
#对某些块应用不同的初始化方法


In [19]:
def init_xavier(m):
    if type(m) == nn.Linear:
        nn.init.xavier_uniform_(m.weight)
def init_42(m):
    if type(m) == nn.Linear:
        nn.init.constant_(m.weight, 42)
net[0].apply(init_xavier)
net[2].apply(init_42)
print(net[0].weight.data[0])
print(net[2].weight.data)


tensor([ 0.5854, -0.5727, -0.3650, -0.5464])
tensor([[42., 42., 42., 42., 42., 42., 42., 42.]])


In [20]:
# 自定义初始化参数
def my_init(m):
    if type(m)==nn.Linear:
        print("Init", *[(name, param.shape)
                        for name, param in m.named_parameters()][0])
        nn.init.uniform_(m.weight,-10,10)
        m.weight.data*= m.weight.data.abs()>=5
net.apply(my_init)
net[0].weight[:2]


Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[ 0.0000, -0.0000,  0.0000, -8.1370],
        [-9.0240, -0.0000,  0.0000, -5.5231]], grad_fn=<SliceBackward0>)

In [21]:
#注意，我们始终可以直接设置参数。
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]


tensor([42.0000,  1.0000,  1.0000, -7.1370])

In [22]:
## 参数绑定- 共享参数


In [23]:
# 我们需要给共享层一个名称，以便可以引用它的参数
shared = nn.Linear(8, 8)
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.Linear(8, 1))
net(X)
# 检查参数是否相同
print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
# 确保它们实际上是同一个对象，而不只是有相同的值
print(net[2].weight.data[0] == net[4].weight.data[0])


tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])
